# Setup

In [46]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [47]:
import os
load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv('OpenAI_API_KEY')

os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

MODEL = "gpt-4o-mini"
openai = OpenAI()

In [48]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = InMemoryVectorStore(embeddings)

In [49]:
import pandas as pd
from langchain_core.documents.base import Document

sheets = pd.read_excel("C&E.xlsm", sheet_name=None)

doc = []
for sheet_name, df in sheets.items():
    df = df.fillna("")
    for idx, row in df.iterrows():
        text = " | ".join(map(str, row.values))
        doc.append(
            Document(
                page_content=text,
                metadata={"sheet": sheet_name, "row": idx + 2}
            )
        )


In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(doc)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 6473 sub-documents.


In [51]:
document_ids = vector_store.add_documents(documents=all_splits)

# Tools

In [73]:
system_message = """
You make the incoming messsage readable.
The format of the response should be as follows:
Sheet: <sheet name>
Row: <row number>
Columns: <column values>
"""

In [53]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [75]:
from langchain.agents import create_agent

tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a excel file. "
    "The excel file is a cause-and-effect diagram for fire and gas"
    "Use the tool to help answer user queries."
    "If you can't find the exact match, use your best judgement to provide a helpful answer."
    "Also mention the sheet name, row number and the column letter (add + 1 to the letter for example if the column is A --> it is B) from which the information was retrieved."
    "If you can't find the information, just say it isnt't available."
)
agent = create_agent(MODEL, tools, system_prompt=prompt)

# Conversational part


In [76]:
def agent_excel(query: str):
    result = agent.invoke(
        {"messages": [
            {"role": "user", "content": query}
            ]
        }
    )
    return result["messages"][-1].content


In [77]:
def chat(message, history):
    agent_response = agent_excel(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": agent_response}]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=512,
)
    response_text = response.choices[0].message.content
    return response_text

In [78]:
view = gr.ChatInterface(
    fn=chat,
    type="messages",
)

view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
